# Podz Liability Fund — CNN + Fusion Training
Runs on Google Colab (free T4 GPU). Reads frames and pose CSVs from Google Drive.
Expected runtime: ~40 min CNN + ~15 min fusion on T4.

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Try multiple possible paths (shortcut, shared, or direct)
candidates = [
    '/content/drive/MyDrive/podz-liability-fund',
    '/content/drive/MyDrive/Podz-Liability-Fund',
    '/content/drive/Shareddrives/podz-liability-fund',
]
GDRIVE = next((p for p in candidates if os.path.exists(p)), None)
if GDRIVE is None:
    raise FileNotFoundError("Could not find podz-liability-fund on Drive. Make sure you've added a shortcut to My Drive from 'Shared with me'.")
print('Drive mounted. Using:', GDRIVE)
print('Contents:', os.listdir(GDRIVE))

## 2. Clone repo and install dependencies

In [ ]:
import os, sys

REPO_DIR = '/content/podz-liability-fund'

if not os.path.exists(REPO_DIR):
    !git clone -b feat/fusion https://github.com/Nathan-Dela-Pena/Podz-Liability-Fund.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout feat/fusion && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

!pip install -q -r requirements.txt
!pip install -q ultralytics
print('Setup complete. Branch:', os.popen(f'cd {REPO_DIR} && git branch --show-current').read().strip())

## 3. Configure paths to point at Google Drive

In [ ]:
# Override config paths before importing any project modules
import config

GDRIVE = '/content/drive/MyDrive/podz-liability-fund'
config.FRAMES_DIR     = f'{GDRIVE}/frames'
config.POSE_DIR       = f'{GDRIVE}/pose'
config.CHECKPOINTS_DIR = f'{GDRIVE}/checkpoints'

os.makedirs(config.CHECKPOINTS_DIR, exist_ok=True)

# Verify
print('FRAMES_DIR:     ', config.FRAMES_DIR)
print('POSE_DIR:       ', config.POSE_DIR)
print('CHECKPOINTS_DIR:', config.CHECKPOINTS_DIR)

import glob
n_frames = len(glob.glob(f'{config.FRAMES_DIR}/**/*.jpg', recursive=True))
n_pose   = len(glob.glob(f'{config.POSE_DIR}/*.csv'))
print(f'Frames: {n_frames} | Pose CSVs: {n_pose}')

## 4. Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

## 5. Train CNN branch (~40 min on T4)

In [ ]:
from src.models.cnn import train_cnn
import os

train_csv = os.path.join(REPO_DIR, 'data/processed/train.csv')
val_csv   = os.path.join(REPO_DIR, 'data/processed/val.csv')

print('train.csv exists:', os.path.exists(train_csv))
print('val.csv exists:  ', os.path.exists(val_csv))

cnn_model = train_cnn(train_path=train_csv, val_path=val_csv, device=device)
print('CNN training complete. Checkpoint saved to', config.CHECKPOINTS_DIR)

## 6. Train fusion model (~15 min on T4)
Requires `lstm_best.pt` in CHECKPOINTS_DIR (copy from local checkpoints/ folder to Drive).

In [ ]:
lstm_ckpt = f'{config.CHECKPOINTS_DIR}/lstm_best.pt'
cnn_ckpt  = f'{config.CHECKPOINTS_DIR}/cnn_best.pt'

print('lstm_best.pt present:', os.path.exists(lstm_ckpt))
print('cnn_best.pt  present:', os.path.exists(cnn_ckpt))

if not os.path.exists(lstm_ckpt):
    print('\nUpload lstm_best.pt to your Drive checkpoints folder, then re-run.')

In [ ]:
from src.models.fusion import train_fusion

fusion_model = train_fusion(
    train_path=os.path.join(REPO_DIR, 'data/processed/train.csv'),
    val_path=os.path.join(REPO_DIR, 'data/processed/val.csv'),
    device=device,
)
print('Fusion training complete. Checkpoints saved to', config.CHECKPOINTS_DIR)

## 7. Download checkpoints
Checkpoints are saved directly to Google Drive — no download needed.
Find them at `My Drive/podz-liability-fund/checkpoints/`.

In [ ]:
print('Saved checkpoints:')
for f in sorted(glob.glob(f'{config.CHECKPOINTS_DIR}/*.pt')):
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)}  ({size_mb:.1f} MB)')